# Multilingual Maternal Health Assistant
**Language auto-detect + translation → MedGemma maternal-health chatbot**

Supported languages: English, Nyanja, Bemba, Lozi, Tonga, Shona, Ndebele

This notebook builds a small web app (in the notebook output) with two tabs:
- **Translator**: translate between the 7 languages, with automatic detection of the input language.
- **Chatbot**: ask maternal-health questions in any of the 7 languages. Questions are answered by MedGemma with the `dawahealth/medgemma-maternal-lora-v2` adapter.

**How the pieces fit together**
| Component | Model | Role |
|---|---|---|
| Language detection | GlotLID (fastText) | Identifies which of the 7 languages the user wrote in |
| Translation | NLLB-200 (default), TranslateGemma (optional) | Nyanja, Bemba, Shona and Ndebele ↔ English |
| Chatbot | MedGemma 4B + maternal LoRA | Answers the question |

**Language coverage**
- **Nyanja, Bemba, Shona:** translated by NLLB-200.
- **Ndebele:** NLLB-200 has no Ndebele, so Zulu is used as a close approximation.
- **Lozi, Tonga:** no translation model in this notebook supports them, so translation is reported as unavailable instead of guessed. The chatbot can still answer in them directly through the adapter, but quality is unverified.
- **TranslateGemma** accepts some of these language codes, but its output for them is unreliable, so it is loaded only as an optional engine for comparison.

**Chatbot modes**
- **Auto (default):** Shona, Nyanja and Bemba questions are translated to English, answered by MedGemma, and translated back, because the adapter's own free-text answers in those languages are weak. Ndebele, Lozi, Tonga and English are answered directly.
- **Direct:** MedGemma answers in the user's language for every language.
- **Always via English:** translate, answer in English, translate back, for every language that has a translator.

**Requirements:** a GPU runtime (T4 or better) and a free Hugging Face account.

**Important:** this is a research tool, not a medical device. It can give incorrect medical information. Have clinicians and native speakers review its outputs before any patient-facing use. NLLB-200 is licensed for non-commercial use (CC-BY-NC), and the adapter is AGPL-3.0 (see its model card).

## Install
Run the next cell, then **Runtime → Restart session** so the newly installed packages are picked up. After the restart, skip the install cell and continue from the token cell.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes "huggingface_hub<2" sentencepiece
!pip install -q gradio fasttext-predict

## How to get a Hugging Face token
1. Create a free account at https://huggingface.co/join (or log in).
2. Open the two model pages and click **Agree and access repository** on each:
   - https://huggingface.co/google/translategemma-4b-it
   - https://huggingface.co/google/medgemma-4b-it
3. Go to https://huggingface.co/settings/tokens → **Create new token**.
4. Choose token type **Read** (simplest). If you pick *Fine-grained*, tick "Read access to contents of all public gated repos you can access".
5. Name it (e.g. `colab`), click **Create token**, and **copy it right away**. It starts with `hf_` and is only shown once.
6. In Colab, click the **key icon (Secrets)** in the left sidebar → **Add new secret**.
   - Name: `HF_TOKEN`
   - Value: the token you copied
   - Turn on **Notebook access**.
7. Run the next cell. If no secret is found, it will ask you to paste the token instead.

Never share your token or paste it into a cell you plan to publish.

In [ ]:
import os, re, gc, torch
from huggingface_hub import login, hf_hub_download

# ---------- Hugging Face login ----------
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face token: ")
login(token=HF_TOKEN)

# ---------- Config ----------
TRANSLATEGEMMA_ID = "google/translategemma-4b-it"
NLLB_ID           = "facebook/nllb-200-distilled-1.3B"
MEDGEMMA_BASE     = "google/medgemma-4b-it"
MEDGEMMA_ADAPTER  = "dawahealth/medgemma-maternal-lora-v2"

LOAD_TRANSLATEGEMMA = True   # optional experimental engine; set False to save ~3 GB of GPU memory
PIVOT_VIA_ENGLISH   = True   # non-English -> non-English goes through English
DETECT_MIN_CONF     = 0.35
# In "Auto" chat mode these languages are answered in English by MedGemma and translated back, because the
# adapter's own free-text answers in them are weak (see its model card).
PIVOT_LANGS         = {"Shona", "Nyanja", "Bemba"}
MAX_HISTORY_TURNS   = 1      # the adapter was trained on single-turn examples; long history can hurt accuracy
MAX_NEW_TOKENS_CHAT = 400
GUIDELINES_DIR      = "/content/guidelines"   # optional: put vetted .txt/.md/.pdf guideline files here

# GlotLID labels used for auto-detect
LANGUAGES = {
    "English": {"iso3": "eng", "glot": ["eng_Latn"]},
    "Nyanja":  {"iso3": "nya", "glot": ["nya_Latn"]},
    "Bemba":   {"iso3": "bem", "glot": ["bem_Latn"]},
    "Lozi":    {"iso3": "loz", "glot": ["loz_Latn"]},
    "Tonga":   {"iso3": "toi", "glot": ["toi_Latn", "tog_Latn"]},
    "Shona":   {"iso3": "sna", "glot": ["sna_Latn"]},
    # Zimbabwean Ndebele is often labelled Zulu / S. Ndebele by language-ID models, so they are merged here
    "Ndebele": {"iso3": "nde", "glot": ["nde_Latn", "nbl_Latn", "zul_Latn"]},
}
LANG_NAMES = list(LANGUAGES)
GLOT_TO_LANG = {lab: name for name, c in LANGUAGES.items() for lab in c["glot"]}

# Default translation engine per language. None = no reliable translator available here.
ENGINE_FOR = {"English": "any", "Nyanja": "nllb", "Bemba": "nllb", "Shona": "nllb",
              "Ndebele": "nllb",   # NLLB has no Ndebele; Zulu is used as a close approximation
              "Lozi": None, "Tonga": None}
NLLB_CODES = {"English": "eng_Latn", "Nyanja": "nya_Latn", "Bemba": "bem_Latn",
              "Shona": "sna_Latn", "Ndebele": "zul_Latn"}

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime!)")
print("Compute dtype:", compute_dtype)

# ---------- Environment checks (fail early with a clear message) ----------
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
                       "then Runtime > Restart session and run the notebook again from the top.")
try:
    import bitsandbytes as bnb
    print("bitsandbytes", bnb.__version__)
except Exception as e:
    raise RuntimeError("bitsandbytes is not importable (" + repr(e) + "). "
                       "Run the install cell again, then Runtime > Restart session, "
                       "and continue from the token cell WITHOUT re-running the install cell.")

## 1. Load NLLB-200 (default translator)
NLLB-200 is Meta's translation model covering 200 languages, including Nyanja, Bemba, Shona and Zulu. The cell also verifies that each language code exists in the tokenizer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nllb_tok = AutoTokenizer.from_pretrained(NLLB_ID)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_ID, torch_dtype=torch.float16).to("cuda").eval()

# Verify every language code really exists in this tokenizer
for lang, c in NLLB_CODES.items():
    tid = nllb_tok.convert_tokens_to_ids(c)
    print(f"{lang:8} {c:9} token id {tid}", "OK" if tid != nllb_tok.unk_token_id else "MISSING")
    if tid == nllb_tok.unk_token_id:
        raise RuntimeError(f"NLLB tokenizer has no code {c}")

## 2. Load TranslateGemma (optional, experimental)
This model is loaded so you can compare it with NLLB-200 in the Translator tab. The table shows whether its chat template *accepts* each language code. Accepting a code does not guarantee good translations: it is documented for 55 languages, and quality for the languages here is unverified.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True,
)

tg_processor = tg_model = None
TG_PLAN = {}
if LOAD_TRANSLATEGEMMA:
    tg_processor = AutoProcessor.from_pretrained(TRANSLATEGEMMA_ID)
    tg_kwargs = {"device_map": "auto"}
    if not use_bf16:   # fp16 can be unstable for Gemma 3 on T4, so use 4-bit there
        tg_kwargs["quantization_config"] = bnb_config
    else:
        tg_kwargs["torch_dtype"] = compute_dtype
    tg_model = AutoModelForImageTextToText.from_pretrained(TRANSLATEGEMMA_ID, **tg_kwargs).eval()

    TG_CODES = {"English": ["en"], "Nyanja": ["ny"], "Bemba": ["bem"], "Lozi": ["loz"],
                "Tonga": ["toi"], "Shona": ["sn"], "Ndebele": ["nd"]}
    def code_supported(code):
        try:
            tg_processor.apply_chat_template(
                [{"role": "user", "content": [{"type": "text", "source_lang_code": code,
                                               "target_lang_code": "en", "text": "hello"}]}],
                tokenize=False, add_generation_prompt=True)
            return True
        except Exception:
            return False
    for name, codes in TG_CODES.items():
        ok = next((c for c in codes if code_supported(c)), None)
        TG_PLAN[name] = ("template", ok) if ok else ("prompt", LANGUAGES[name]["iso3"])
    print(f"{'Language':10} {'TranslateGemma route':22} Code")
    for n, (route, c) in TG_PLAN.items():
        print(f"{n:10} {'template accepts code' if route=='template' else 'unofficial prompt':22} {c}")
else:
    print("TranslateGemma skipped (LOAD_TRANSLATEGEMMA = False)")

## 3. Translation functions
A small router picks the engine for each language pair, translates through English when neither side is English, and rejects output written in the wrong script. If a language has no reliable translator, it raises a clear "unavailable" message.

In [ ]:
class TranslationUnavailable(Exception):
    pass

def is_latin_script(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return True
    return sum(ord(ch) < 0x250 or 0x1E00 <= ord(ch) <= 0x1EFF for ch in letters) / len(letters) >= 0.9

# ---------- NLLB-200 ----------
def _split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]

@torch.inference_mode()
def _nllb_batch(sentences, src_code, tgt_code):
    nllb_tok.src_lang = src_code
    enc = nllb_tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
    out = nllb_model.generate(**enc, forced_bos_token_id=nllb_tok.convert_tokens_to_ids(tgt_code),
                              num_beams=4, max_new_tokens=256, max_length=None, no_repeat_ngram_size=4)
    return nllb_tok.batch_decode(out, skip_special_tokens=True)

def _nllb_translate(text, s, t):
    lines = []
    for line in text.split("\n"):
        if not line.strip():
            lines.append(line); continue
        sents = _split_sentences(line)
        res = []
        for i in range(0, len(sents), 8):
            res += _nllb_batch(sents[i:i+8], NLLB_CODES[s], NLLB_CODES[t])
        lines.append(" ".join(res))
    return "\n".join(lines)

# ---------- TranslateGemma (experimental) ----------
@torch.inference_mode()
def _tg_generate(inputs, max_new_tokens=512):
    inputs = inputs.to(tg_model.device)
    n = inputs["input_ids"].shape[-1]
    out = tg_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tg_processor.decode(out[0][n:], skip_special_tokens=True).strip()

def _tg_line(text, s, t):
    (sk, sc), (tk, tc) = TG_PLAN[s], TG_PLAN[t]
    if sk == "template" and tk == "template":
        msgs = [{"role": "user", "content": [{"type": "text", "source_lang_code": sc,
                                              "target_lang_code": tc, "text": text}]}]
        return _tg_generate(tg_processor.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"))
    prompt = (f"You are a professional {s} ({sc}) to {t} ({tc}) translator. Your goal is to accurately convey "
              f"the meaning and nuances of the original {s} text while adhering to {t} grammar, vocabulary, "
              f"and cultural sensitivities.\nProduce only the {t} translation, without any additional "
              f"explanations or commentary. Please translate the following {s} text into {t}:\n\n\n{text}")
    full = f"<bos><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    return _tg_generate(tg_processor.tokenizer(full, add_special_tokens=False, return_tensors="pt"))

def _tg_translate(text, s, t):
    if tg_model is None:
        raise TranslationUnavailable("TranslateGemma is not loaded (LOAD_TRANSLATEGEMMA = False).")
    return "\n".join(_tg_line(l, s, t) if l.strip() else l for l in text.split("\n"))

# ---------- Router ----------
def _hop(text, s, t, engine):
    other = t if s == "English" else s
    eng = engine
    if engine == "auto":
        eng = ENGINE_FOR[other]
        if eng is None:
            raise TranslationUnavailable(
                f"No reliable translator for {other} is available in this notebook. "
                f"(NLLB-200 does not cover it, and TranslateGemma is not reliable for it.)")
    if eng == "nllb":
        if other not in NLLB_CODES:
            raise TranslationUnavailable(f"NLLB-200 does not cover {other}.")
        out = _nllb_translate(text, s, t)
        note = "NLLB-200" + (" (Ndebele approximated by Zulu)" if other == "Ndebele" else "")
    elif eng == "translategemma":
        out = _tg_translate(text, s, t)
        note = "TranslateGemma (experimental, unverified for this language)"
    else:
        raise TranslationUnavailable(f"Unknown engine {eng}")
    if not is_latin_script(out):
        raise TranslationUnavailable(f"Output rejected: the model produced the wrong script for {t}.")
    return out, note

def translate_ex(text, src, tgt, engine="auto"):
    """engine: auto | nllb | translategemma. Returns (translation, note). Raises TranslationUnavailable."""
    if src == tgt:
        return text, "same language, nothing to do"
    hops = [(src, tgt)]
    if PIVOT_VIA_ENGLISH and src != "English" and tgt != "English":
        hops = [(src, "English"), ("English", tgt)]
    cur, notes = text, []
    for s, t in hops:
        cur, n = _hop(cur, s, t, engine)
        notes.append(n)
    return cur, ("via English; " if len(hops) == 2 else "") + "; ".join(dict.fromkeys(notes))

def translate(text, src, tgt, engine="auto"):
    return translate_ex(text, src, tgt, engine)[0]

def can_translate(lang):
    return lang == "English" or ENGINE_FOR.get(lang) is not None

## 4. Language auto-detect (GlotLID)
GlotLID is a fastText language-ID model. Its predictions are restricted to the 7 supported languages, and low-confidence results are reported as "unsure".

In [ ]:
import fasttext
glot = fasttext.load_model(hf_hub_download("cis-lmu/glotlid", "model.bin"))

def detect_language(text, k=40):
    """Returns (language_name or None, confidence, scores) restricted to the 7 supported languages."""
    t = " ".join(text.split())
    if not t:
        return None, 0.0, {}
    labels, probs = glot.predict(t, k=k)
    scores = {}
    for label, prob in zip(labels, probs):
        name = GLOT_TO_LANG.get(label.replace("__label__", ""))
        if name:
            scores[name] = scores.get(name, 0.0) + float(prob)
    if not scores:
        return None, 0.0, {}
    best = max(scores, key=scores.get)
    return (best if scores[best] >= DETECT_MIN_CONF else None), scores[best], scores

for s in ["Where is the nearest clinic?", "Ndiri kuda kuziva zvekubereka", "Ngicela ukwazi ngokukhulelwa",
          "Nikwata ubwafya bwa mala"]:
    print(s, "->", detect_language(s)[:2])

## 5. Round-trip test (English → language → English)
Each language is translated from English and then back to English. Have a native speaker check the translated text. If the round trip no longer means the original, the translation is not good enough. To compare engines, change `engine` to `"translategemma"` in the cell below.

In [ ]:
sample = "Please go to the clinic if you have a severe headache or bleeding during pregnancy."
print("ORIGINAL:", sample, "\n")
for lang in LANG_NAMES[1:]:
    try:
        fwd, note = translate_ex(sample, "English", lang, engine="auto")
        back = translate(fwd, lang, "English", engine="auto")
        print(f"[{lang}] {fwd}\n   back to English: {back}\n   ({note})\n")
    except TranslationUnavailable as e:
        print(f"[{lang}] UNAVAILABLE: {e}\n")

## 6. Load the MedGemma maternal-health chatbot
Base: `google/medgemma-4b-it` (4-bit) + LoRA `dawahealth/medgemma-maternal-lora-v2`.

In [ ]:
from peft import PeftModel
from contextlib import nullcontext

gc.collect(); torch.cuda.empty_cache()

med_processor = AutoProcessor.from_pretrained(MEDGEMMA_BASE)
med_base = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_BASE, quantization_config=bnb_config, device_map="auto")
med_model = PeftModel.from_pretrained(med_base, MEDGEMMA_ADAPTER).eval()
print("MedGemma + LoRA loaded.")

# System prompt from the adapter's model card. The adapter was trained with it,
# so changing the wording can reduce accuracy. It answers in whatever language the question is in.
SYSTEM_NATIVE = ("You are a maternal health assistant. Answer in the same language and mixing of languages "
                 "the person uses. Give clear, supportive guidance, and tell the person to go to a clinic or "
                 "hospital right away if there are danger signs.")
SYSTEM_EN = SYSTEM_NATIVE

def build_prompt(system, pairs, question, context=None):
    pairs = pairs[-MAX_HISTORY_TURNS:] if MAX_HISTORY_TURNS > 0 else []
    turns = [(q, a) for q, a in pairs] + [(question, None)]
    parts = []
    for i, (q, a) in enumerate(turns):
        if a is None and context:
            q = f"{context}\n\nQuestion: {q}"
        content = f"{system}\n\n{q}" if i == 0 else q
        parts.append(f"<start_of_turn>user\n{content}<end_of_turn>\n<start_of_turn>model\n")
        if a is not None:
            parts.append(f"{a}<end_of_turn>\n")
    return "".join(parts)

def _trim_incomplete(text):
    # if generation hit the token limit mid-sentence, cut back to the last full sentence
    m = list(re.finditer(r"[.!?](?=\s|$)", text))
    return text[:m[-1].end()] if m and m[-1].end() > len(text) * 0.5 else text

@torch.inference_mode()
def medgemma_generate(system, pairs, question, context=None, use_adapter=True,
                      max_new_tokens=MAX_NEW_TOKENS_CHAT):
    prompt = build_prompt(system, pairs, question, context)
    inputs = med_processor(text=prompt, return_tensors="pt").to(med_model.device)
    ctx = nullcontext() if use_adapter else med_model.disable_adapter()
    with ctx:
        out = med_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                 repetition_penalty=1.15)   # recommended in the model card
    new = out[0][inputs["input_ids"].shape[1]:]
    text = med_processor.decode(new, skip_special_tokens=True).strip()
    return _trim_incomplete(text) if len(new) >= max_new_tokens else text

## 7. Optional: ground answers in vetted guidelines (retrieval)
The adapter's model card notes that outputs can be clinically wrong and recommends grounding answers in vetted guideline content. To use this, put guideline files (`.txt`, `.md` or `.pdf`, ideally English, e.g. national antenatal-care guidelines or WHO material) in `/content/guidelines`, then run the next cell. In the chatbot, tick **Ground answers in guideline files**. This step is optional. The adapter was not trained with reference passages, so compare answers with and without it.

In [ ]:
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

kb = {"chunks": [], "vec": None, "mat": None}

def _read_file(path):
    if path.lower().endswith(".pdf"):
        try:
            from pypdf import PdfReader
        except ImportError:
            get_ipython().system("pip install -q pypdf")
            from pypdf import PdfReader
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    return open(path, encoding="utf-8", errors="ignore").read()

def load_guidelines(folder=GUIDELINES_DIR, words=120, overlap=30):
    os.makedirs(folder, exist_ok=True)
    chunks = []
    for path in sorted(glob.glob(os.path.join(folder, "*"))):
        if not path.lower().endswith((".txt", ".md", ".pdf")):
            continue
        toks = _read_file(path).split()
        for i in range(0, max(len(toks), 1), words - overlap):
            piece = " ".join(toks[i:i + words])
            if len(piece.split()) > 15:
                chunks.append({"text": piece, "source": os.path.basename(path)})
    kb["chunks"] = chunks
    if chunks:
        kb["vec"] = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
        kb["mat"] = kb["vec"].fit_transform([c["text"] for c in chunks])
    print(f"Loaded {len(chunks)} passages from {folder}")

def retrieve(query_en, k=3, min_score=0.08):
    if not kb["chunks"]:
        return []
    sims = linear_kernel(kb["vec"].transform([query_en]), kb["mat"]).ravel()
    return [(kb["chunks"][i], float(sims[i])) for i in sims.argsort()[::-1][:k] if sims[i] >= min_score]

def format_context(hits):
    lines = ["Use the reference information below when it is relevant. If it does not answer the question, "
             "say you are not sure and advise the person to see a health worker.", "", "Reference information:"]
    for n, (c, _) in enumerate(hits, 1):
        lines.append(f"[{n}] ({c['source']}) {c['text']}")
    return "\n".join(lines)

load_guidelines()

## 8. Optional: compare the adapter with base MedGemma
Answers each test question with and without the maternal-health adapter. Edit the questions to ones where you know the correct answer. If the base model is more accurate on a topic, the adapter is the weak point; if both are wrong, retrieval from vetted guidelines or a clinician-reviewed answer set is needed.

In [ ]:
test_questions = [
    "What are the danger signs in pregnancy that mean I must go to the clinic immediately?",
    "How many antenatal visits should I have during pregnancy?",
    "Why is it important to test for HIV during pregnancy?",
]
for q in test_questions:
    print("=" * 80, "\nQ:", q)
    print("\n[WITH adapter]\n", medgemma_generate(SYSTEM_NATIVE, [], q, use_adapter=True))
    print("\n[BASE MedGemma, no adapter]\n", medgemma_generate(SYSTEM_NATIVE, [], q, use_adapter=False))

In [ ]:
# ---------- Pipeline: detect -> (optional translate / retrieval) -> MedGemma ----------
def new_state():
    return {"native": [], "en": []}

def chat_turn(user_text, state, mode="native", show_english=False, use_adapter=True, use_rag=False):
    lang, conf, _ = detect_language(user_text)
    known = lang is not None
    notes = [f"Detected: **{lang}** ({conf:.0%})" if known else "Language: unsure"]
    notes.append("adapter on" if use_adapter else "base MedGemma (adapter off)")
    if mode == "auto":
        mode = "pivot" if (known and lang in PIVOT_LANGS) else "native"
        notes.append(f"auto → {'translate via English' if mode == 'pivot' else 'direct'}")

    # English version of the question, needed for pivot mode and for retrieval
    q_en = user_text if (not known or lang == "English") else None
    if q_en is None and (mode == "pivot" or use_rag):
        if can_translate(lang):
            try:
                q_en = translate(user_text, lang, "English")
            except TranslationUnavailable as e:
                notes.append(f"translation failed ({e})")
        else:
            notes.append(f"no reliable translator for {lang}")

    context = None
    if use_rag:
        if not kb["chunks"]:
            notes.append("no guideline files loaded")
        elif q_en:
            hits = retrieve(q_en)
            context = format_context(hits) if hits else None
            notes.append(f"{len(hits)} guideline passage(s) used")
        else:
            notes.append("retrieval skipped (no English query)")

    reply_en = None
    if mode == "pivot" and known and lang != "English" and q_en is not None:
        reply_en = medgemma_generate(SYSTEM_EN, state["en"], q_en, context, use_adapter)
        try:
            reply = translate(reply_en, "English", lang)
            notes.append("translate → MedGemma → translate back")
        except TranslationUnavailable as e:
            reply = reply_en
            notes.append(f"could not translate reply back ({e}); showing English")
        state["en"].append((q_en, reply_en))
    else:
        reply = medgemma_generate(SYSTEM_NATIVE, state["native"], user_text, context, use_adapter)
        notes.append("MedGemma answered directly")
    state["native"].append((user_text, reply))

    shown = reply
    if show_english and known and lang != "English":
        try:
            eng = reply_en if reply_en is not None else translate(reply, lang, "English")
            shown += f"\n\n*English check:* {eng}"
        except TranslationUnavailable:
            shown += f"\n\n*(English check not available for {lang})*"
    return shown, " · ".join(notes), state

## 9. Launch the app (Translator tab + Chatbot tab)
Run the cell and open the link it prints. In the Chatbot tab you can choose the mode, show an English version of each reply for checking, switch the adapter on or off, and enable guideline grounding.

In [ ]:
import gradio as gr

ENGINE_CHOICES = {"Auto (recommended)": "auto", "NLLB-200": "nllb", "TranslateGemma (experimental)": "translategemma"}

def ui_translate(text, src, tgt, engine_label):
    if not text.strip():
        return "", ""
    head = ""
    if src == "Auto-detect":
        lang, conf, _ = detect_language(text)
        if lang is None:
            return "", "Could not confidently detect the language. Pick it manually."
        src, head = lang, f"Detected **{lang}** ({conf:.0%}). "
    try:
        out, note = translate_ex(text, src, tgt, ENGINE_CHOICES[engine_label])
        return out, head + "Engine: " + note
    except TranslationUnavailable as e:
        return "", head + "⚠️ " + str(e)

def ui_chat(user_text, chat, state, mode, show_en, use_adapter, use_rag):
    if not user_text.strip():
        return "", chat, state, ""
    mode_key = "auto" if mode.startswith("Auto") else ("pivot" if mode.startswith("Translate") else "native")
    reply, info, state = chat_turn(user_text, state, mode_key, show_en, use_adapter, use_rag)
    chat = chat + [{"role": "user", "content": user_text}, {"role": "assistant", "content": reply}]
    return "", chat, state, info

def ui_clear():
    return [], new_state(), ""

with gr.Blocks(title="Maternal Health Assistant") as demo:
    gr.Markdown("# Maternal Health Assistant\nEnglish · Nyanja · Bemba · Lozi · Tonga · Shona · Ndebele\n\n"
                "**Not a medical device.** For danger signs (heavy bleeding, severe headache, fits, "
                "reduced baby movement, fever) go to a clinic or hospital immediately.")
    with gr.Tab("Translator"):
        gr.Markdown("Machine translation for these languages is imperfect. Lozi and Tonga are not supported, "
                    "and Ndebele is approximated by Zulu. Have a native speaker check anything important.")
        with gr.Row():
            src = gr.Dropdown(["Auto-detect"] + LANG_NAMES, value="Auto-detect", label="From")
            tgt = gr.Dropdown(LANG_NAMES, value="English", label="To")
            eng = gr.Dropdown(list(ENGINE_CHOICES), value="Auto (recommended)", label="Engine")
        t_in = gr.Textbox(lines=5, label="Text")
        t_btn = gr.Button("Translate", variant="primary")
        t_out = gr.Textbox(lines=5, label="Translation")
        t_info = gr.Markdown()
        t_btn.click(ui_translate, [t_in, src, tgt, eng], [t_out, t_info])
    with gr.Tab("Chatbot"):
        MODE_AUTO = "Auto (recommended): via English for Shona/Nyanja/Bemba, direct otherwise"
        MODE_A = "MedGemma answers directly"
        MODE_B = "Translate → MedGemma → translate back (always)"
        mode = gr.Radio([MODE_AUTO, MODE_A, MODE_B], value=MODE_AUTO, label="Mode")
        show_en = gr.Checkbox(True, label="Show English version of the reply (for checking)")
        use_ad = gr.Checkbox(True, label="Use the maternal-health LoRA adapter (untick to compare with base MedGemma)")
        use_rg = gr.Checkbox(False, label="Ground answers in guideline files (needs files in /content/guidelines)")
        try:
            bot = gr.Chatbot(type="messages", height=420, label="MedGemma maternal health")
        except TypeError:   # newer Gradio versions only support the messages format
            bot = gr.Chatbot(height=420, label="MedGemma maternal health")
        st = gr.State(new_state())
        box = gr.Textbox(placeholder="Type in any of the 7 languages...", label="Your message")
        info = gr.Markdown()
        with gr.Row():
            send = gr.Button("Send", variant="primary")
            clear = gr.Button("Clear")
        box.submit(ui_chat, [box, bot, st, mode, show_en, use_ad, use_rg], [box, bot, st, info])
        send.click(ui_chat, [box, bot, st, mode, show_en, use_ad, use_rg], [box, bot, st, info])
        clear.click(ui_clear, None, [bot, st, info])

demo.queue().launch(debug=False)